## Control flow — `if/else`, `for_each`, retries

Three control-flow primitives the exam can plant a question on.

**`if/else` condition task.** A task whose only job is to evaluate a boolean and route the DAG. Downstream tasks declare "only run if my parent's branch was `true`":

```text
                        ┌─ true ─► run_fraud_rebuild
  is_friday ─► if/else ─┤
                        └─ false ► skip_to_dashboard
```

**`for_each` task.** Runs a child task once per item in an array — fan the same notebook out over many inputs:

```text
  inputs = ["cards", "accounts", "loans", "payments"]
  child notebook ingest.py runs 4×, once per vertical, in parallel
```

The child receives one item per execution as a parameter (`{{input}}` in templates).

**Retries — configured at the task level:**

- **`max_retries`** — times to re-run on failure. **`min_retry_interval_millis`** — minimum gap between attempts.
- **`retry_on_timeout`** — also retry on timeout, not just error. **`timeout_seconds`** — kill the task if it runs longer.

**The exam pattern:** a flaky upstream API fails ~1% of the time. Set `max_retries = 3` on the ingest task — the transient failure auto-retries; only a *persistent* failure pages a human. Don't reach for `for_each` or a continuous trigger for retry — retry is its own task setting.
